# Machine Learning LAB 5: Random Forests

Course 2025/26: *F. Chiariotti*

The notebook contains a simple learning task over which we will implement a **RANDOM FOREST**.

Complete all the **required code sections**.

### IMPORTANT for the exam:

The functions you might be required to implement in the exam will have the same signature and parameters as the ones in the labs

## Classification of Stayed/Churned Customers

The Customer Churn table contains information on all 3,758 customers from a Telecommunications company in California in Q2 2022. Companies are naturally interested in churn, i.e., in which users are likely to switch to another company soon to get a better deal, and which are more loyal customers.

The dataset contains three features:
- **Tenure in Months**: Number of months the customer has stayed with the company
- **Monthly Charge**: The amount charged to the customer monthly
- **Age**: Customer's age

The aim of the task is to predict if a customer will churn or not based on the three features.

---

## Import all the necessary Python libraries and load the dataset

### The Dataset
The dataset is a `.csv` file containing three input features and a label. Here is an example of the first 4 rows of the dataset: 

<center>

Tenure in Months | Monthly Charge | Age | Customer Status |
| -----------------| ---------------|-----|-----------------|
| 9 | 65.6 | 37 | 0 |
| 9 | -4.0 | 46 | 0 |
| 4 | 73.9 | 50 | 1 |
| ... | ... | ... | ... |

</center>

Customer Status is 0 if the customer has stayed with the company and 1 if the customer has churned.

In [26]:
import numpy as np
import pandas as pd
import random as rnd
from matplotlib import pyplot as plt
from sklearn import linear_model, preprocessing
from sklearn.model_selection import train_test_split

np.random.seed(1)

def load_dataset(filename):
    data_train = pd.read_csv(filename)
    #permute the data
    data_train = data_train.sample(frac=1).reset_index(drop=True) # shuffle the data
    X = data_train.iloc[:, 0:3].values # Get first two columns as the input
    Y = data_train.iloc[:, 3].values # Get the third column as the label
    Y = 2*Y-1 # Make sure labels are -1 or 1 (0 --> -1, 1 --> 1)
    return X,Y

# Load the dataset
X, Y = load_dataset('data/telecom_customer_churn_cleaned.csv')

We are going to differentiate (classify) between **class "1" (churned)** and **class "-1" (stayed)**

## Divide the data into training and test sets

In [27]:
# Compute the splits
m_training = int(0.75*X.shape[0])

# m_test is the number of samples in the test set (total-training)
m_test =  X.shape[0] - m_training
X_training =  X[:m_training]
Y_training =  Y[:m_training]
X_test =   X[m_training:]
Y_test =  Y[m_training:]

print("Number of samples in the train set:", X_training.shape[0])
print("Number of samples in the test set:", X_test.shape[0])
print("Number of churned users in test:", np.sum(Y_test==-1))
print("Number of loyal users in test:", np.sum(Y_test==1))

# Standardize the input matrix
# The transformation is computed on training data and then used on all the 3 sets
scaler = preprocessing.StandardScaler().fit(X_training) 

np.set_printoptions(suppress=True) # sets to zero floating point numbers < min_float_eps
X_training =  scaler.transform(X_training)
print ("Mean of the training input data:", X_training.mean(axis=0))
print ("Std of the training input data:",X_training.std(axis=0))

X_test =  scaler.transform(X_test)
print ("Mean of the test input data:", X_test.mean(axis=0))
print ("Std of the test input data:", X_test.std(axis=0))
print(Y_training)

Number of samples in the train set: 2817
Number of samples in the test set: 940
Number of churned users in test: 479
Number of loyal users in test: 461
Mean of the training input data: [-0.  0. -0.]
Std of the training input data: [1. 1. 1.]
Mean of the test input data: [0.0575483  0.05550169 0.0073833 ]
Std of the test input data: [0.98593187 0.97629659 1.00427583]
[-1  1  1 ...  1 -1 -1]


We will use **homogeneous coordinates** to describe all the coefficients of the model.

_Hint:_ The conversion can be performed with the function $hstack$ in $numpy$.

In [28]:
def to_homogeneous(X_training, X_test):
    Xh_training = np.hstack([np.ones( (X_training.shape[0], 1) ), X_training]) 
    Xh_test = np.hstack([np.ones( (X_test.shape[0], 1) ), X_test])
    return Xh_training, Xh_test

## Decision tree

Now **complete** the class *Tree* and all auxiliary functions. <br>

The input parameters to pass to the *id3_training* function are:
- $X$: the matrix of input features, one row for each sample
- $Y$: the vector of labels for the input features matrix X
- $max\_depth$: the maximum depth of the tree

---> Example:
```python 
import numpy as np
#a=np.array([[3,4,1],[4,2,-1],[4,2,-1],[4,2,-1]])
#left=np.where(a[:,2]<1)
#print(left[0])
#ar=[[1,2,3],[4,6,2]]
#len(ar)
argh=np.array([[1.1,2.1,3.1],[4.1,3.1,7.1],[5.1,2.1,8.1],[5.1,2.1,8.1],[5.1,2.1,8.1],[5.1,2.1,8.1]])
y=np.array([1,-1,-1,1,-1,1])
print(argh[:,0])
print(y)
mask=(argh[:,0]>3)
print(argh[mask])
print(y[mask],type(y[mask]))
#print(np.where((argh[:,0]>3),y,np.empty(6)))
#print(np.where((argh[:,0]>3), argh[:,0], np.empty(6)))

In [29]:
class Tree:

    def __init__(self):
        self.idx = -1    # The index of the feature over which you split (no split: -1)
        self.thresh = 0  # The threshold value over which you split (<=: left, >: right)
        self.leaf = 0    # 1 if it is a leaf of class 1, -1 if it is a leaf of class -1,
                         # 0 if it is an internal node
        self.left = []   # Left subtree (empty if it is a leaf)
        self.right = []  # Right subtree (empty if it is a leaf)

    def entropy(left, right):
        # ---> nparrays containing LABELS of points to the left / to the right of a given threshold coordinate
        H = 0
        tot_length = len(left) + len(right)
        left_prob = len(np.where(left > 0)[0]) / len(left)
        if (left_prob > 0):
            H -= len(left) * left_prob * np.log2(left_prob) / tot_length
        if (left_prob < 1):
            H -= len(left) * (1 - left_prob) * np.log2(1 - left_prob) / tot_length
            # ---> if left_prob = 0 (or left_prob = 1): right_prob=1 -> go to right_prob > 0 term (checks out?)
        right_prob = len(np.where(right > 0)[0]) / len(right)
        if (right_prob > 0):
            H -= len(right) * right_prob * np.log2(right_prob) / tot_length
        if (right_prob < 1):
            H -= len(right) * (1 - right_prob) * np.log2(1 - right_prob) / tot_length
        return H

    def classify(self, x):
        ## classify the point x (easy for leaves, you have to go down the tree if the node is internal) ---> single sample
        node=self
        # ---> NB cannot reassign "self" variable; so, use addional variable and directly access class attributes in while/if blocks
        while (node.leaf==0):
            leaf=node.leaf
            threshold=node.thresh
            index=node.idx
            if (x[index]<=threshold): node=node.left
            else: node=node.right
        return node.leaf

    def id3_training(self, X:np.ndarray, Y:np.ndarray, max_depth:int, printing:bool): 
        # ---> NB didnt use "np.where"
        if (np.max(Y) - np.min(Y) < 1e-3):
            self.leaf = np.max(Y) # ---> 0
            if (printing):
                print('Remaining depth: ' + str(max_depth) +
                      ', leaf node (all labels are the same over ' + str(len(Y)) + ' points)')
            return
        # If the maximum depth is 0, the node must be a leaf
        if (max_depth < 1):
            if (printing):
                print('Remaining depth: ' + str(max_depth) +
                      ', leaf node (maximum depth reached, ' + str(len(Y)) + ' points)')
            if (len(np.where(Y > 0)) > len(Y) / 2):
                self.leaf = 1
            else:
                self.leaf = -1
            return
        # Find the best split: iterate over features
        # ---> NB the correct order (over all features)
        best_idx = -1
        best_thresh = -1
        best_entropy = 1e9
        #n_features=X.shape[1] # ---> maybe this messed up everything bc recursion?
        #indexes_array=[]
        ## Iterate over the features and threshold values
        for feature_idx in range(X.shape[1]):
            #d=0
            #while (d<=max_depth):
            #new_samples=[[X],[]]
            #new_labels=[[Y],[]]
            #splits_array=[]
            #for samples_array,labels_array in zip(new_samples,new_labels):
                #samples=samples_array
                #labels=labels_array
            #for idx in range(samples.shape[0]):
            values = X[:, feature_idx]
            sorted_ind = np.argsort(values)
            values = np.unique(values[sorted_ind]) # ---> NB this three lines to sort (only for threshold - to review?)
            for idx in range(len(values)-1): # --> NB the "-1", and len(values) actually smaller bc only unique points
                #point=X[point_idx][feature_idx]
                threshold = (values[idx] + values[idx + 1]) / 2
                #mask_left=(samples[:, feature_idx]<=best_thresh)
                mask_left=(X[:, feature_idx]<=threshold) # ---> NB mask more efficient than np.where!
                #mask_right=(samples[:, feature_idx]>best_thresh)
                mask_right=(X[:, feature_idx]>threshold)
                if len(Y[mask_left]) == 0 or len(Y[mask_right]) == 0:
                    print('error!',best_idx,threshold,values)  # ---> NB this case
                #left_samples_labels=new_labels[mask_left]
                # ---> [0] in np.where is essentially meaningless, only depends on function architecture
                #right_samples_labels=new_labels[mask_right]
                #entropy=entropy(left_samples_labels, right_samples_labels)
                entropy=Tree.entropy(Y[mask_left], Y[mask_right])# ---> NB the "Tree."
                #threshold=(samples[idx, feature_idx]+samples[idx+1,feature_idx])/2
                if entropy < best_entropy:
                    best_idx=feature_idx 
                    best_thresh=threshold
                    best_entropy=entropy
        if (best_idx == -1):
            # No valid features: the points are all identical
            self.leaf = np.sign(np.sum(Y))
            if (self.leaf == 0):
                self.leaf = 1
            if (printing): print('Remaining depth: ' + str(max_depth) +
                                 ', leaf node (all inputs are the same over ' +
                                 str(len(Y)) + ' points)')
            return
        #left_samples = samples[mask_left]
        #right_samples = samples[mask_right]
        #new_samples=[[left_samples],[right_samples]] 
        #new_labels=[[left_samples_labels],[right_samples_labels]]
        #self.left=left_samples
        #self.right=right_samples
        #if (len(left_samples)==0):
        #    self.leaf=1
        #    return
        #elif (len(right_samples)==0):
        #    self.leaf=-1
        #    return
        #else:
        #    self.leaf=0
        #    #self=self.self
        #    #splits_array.append(best_idx)
        #    #d=d+1
        new_mask_left = (X[:, best_idx] <= best_thresh)
        new_mask_right = (X[:, best_idx] > best_thresh)
        if (printing):
            print('Remaining depth: ' + str(max_depth) + ', splitting ' + str(len(Y)) +
                  ' elements into ' + str(len(X[new_mask_left])) + ' and ' + str(len(X[new_mask_right])) +
                  ' over feature ' + str(best_idx))
        ## run the next recursive step of ID3 over the left and right subtrees
        self.idx = best_idx
        self.thresh = best_thresh 
        self.left=Tree() # ---> NB this
        self.right=Tree()
        self.left.id3_training(X[new_mask_left], Y[new_mask_left], max_depth - 1, printing) 
        self.right.id3_training(X[new_mask_right], Y[new_mask_right], max_depth - 1, printing) # ---> NB this recursive call
#----------------------------------------------------------------------------------------------------------
    def extra_training(self, X, Y, max_depth, printing):
        if (np.max(Y) - np.min(Y) < 1e-3):
            self.leaf = np.max(Y) # ---> 0
            if (printing):
                print('Remaining depth: ' + str(max_depth) +
                      ', leaf node (all labels are the same over ' + str(len(Y)) + ' points)')
            return
        if (max_depth < 1):
            if (printing):
                print('Remaining depth: ' + str(max_depth) +
                      ', leaf node (maximum depth reached, ' + str(len(Y)) + ' points)')
            if (len(np.where(Y > 0)) > len(Y) / 2):
                self.leaf = 1
            else:
                self.leaf = -1
            return
        best_idx = -1
        best_thresh = -1
        best_entropy = 1e9
        ## Iterate over the features (remember, the threshold value is random)! 
        for feature_idx in range(X.shape[1]):
            values = X[:, feature_idx]
            sorted_ind = np.argsort(values)
            values = np.unique(values[sorted_ind]) 
            # Randomize the split!
            if (len(values) > 1): # ---> NB this
                threshold = rnd.uniform(np.min(values), np.max(values))
                mask_left=(X[:, feature_idx]<=threshold)
                mask_right=(X[:, feature_idx]>threshold)
                #if len(Y[mask_left]) == 0 or len(Y[mask_right]) == 0:
                    #print('error!',best_idx,threshold,values) # ---> not needed bc random threshold (unlikely/impossible to pick exactly the extreme)
                entropy=Tree.entropy(Y[mask_left], Y[mask_right])
                if entropy < best_entropy:
                    best_idx=feature_idx 
                    best_thresh=threshold
                    best_entropy=entropy
        if (best_idx == -1):
            self.leaf = np.sign(np.sum(Y))
            if (self.leaf == 0):
                self.leaf = 1
            if (printing): print('Remaining depth: ' + str(max_depth) +
                                 ', leaf node (all inputs are the same over ' +
                                 str(len(Y)) + ' points)')
            return
        new_mask_left = (X[:, best_idx] <= best_thresh)
        new_mask_right = (X[:, best_idx] > best_thresh)
        if (printing):
            print('Remaining depth: ' + str(max_depth) + ', splitting ' + str(len(Y)) +
                  ' elements into ' + str(len(X[new_mask_left])) + ' and ' + str(len(X[new_mask_right])) +
                  ' over feature ' + str(best_idx))
        ## run the next recursive step of ExTra ID3 over the left and right subtrees
        self.idx = best_idx
        self.thresh = best_thresh 
        self.left=Tree() 
        self.right=Tree()
        self.left.extra_training(X[new_mask_left], Y[new_mask_left], max_depth - 1, printing) 
        self.right.extra_training(X[new_mask_right], Y[new_mask_right], max_depth - 1, printing) 

Now we use the implementation to learn a model from the training data directly. Set a maximum depth of 12.

In [30]:
single_tree = Tree()
single_tree.id3_training(X_training, Y_training, 12, False)

train_loss = 0
for i in range(len(Y_training)):
    predicted = single_tree.classify(X_training[i, :])
    if (Y_training[i] != predicted):
        train_loss += 1 / len(Y_training)
print('Training loss: ' + str(train_loss))

test_loss = 0
for i in range(len(Y_test)):
    predicted = single_tree.classify(X_test[i, :])
    if (Y_test[i] != predicted):
        test_loss += 1 / len(Y_test)
print('Test loss: ' + str(test_loss))

Training loss: 0.19524316648917342
Test loss: 0.3276595744680835


This overfits a lot! Let's try to create a random forest

---> Example
```python 
m=np.array([[1,2,4],[3,2,5],[0,4,1],[1,3,1]])
l=np.array([1,1,-1,1])
ix=np.random.choice(range(m.shape[0]), size=m.shape[0])
ixxx=np.sort(np.random.choice(range(m.shape[1]), size=2, replace=False))
print(ix)
print(ixxx)
midm=m[:,ixxx]
newm=midm[ix,:]
#midl=l[ixxx]
newl=l[ix]
print(newm,newl)

In [ ]:
class Forest:

    def __init__(self, trees:int, n_features, n_samples, max_depth): # ---> NB number of features to keep
        self.forest = []
        self.features = n_features
        self.max_depth = max_depth
        self.samples = n_samples
        for i in range(trees):
            self.forest.append(Tree())

    def classify(self, x):
        # Classify the point through a majority vote
        acc=0
        for tree in self.forest:
            vote=tree.classify(x) # ---> +1 or -1
            acc=acc+vote
        if acc==0: # ---> if same number of votes for each candidate
            acc=rnd.choice([-1,1]) 
        return acc
        
    def train(self, X, Y):
        for tree in self.forest:
            X_train, Y_train = self.bag(X, Y) # ---> NB this call
            tree.id3_training(X_train, Y_train, self.max_depth, False)
        
    def bag(self, X, Y):
        ## implement bagging! Sample data points with replacement
        # ---> choose random indexes
        random_feature_indexes=np.sort(np.random.choice(range(X.shape[1]), size=self.features))
        random_sample_indexes=np.random.choice(range(X.shape[0]), size=X.shape[0])
        X_nf=X[:,random_feature_indexes]
        X_bagged=X_nf[random_sample_indexes,:]
        Y_bagged=Y[random_sample_indexes] # ---> review efficiency of this? also, new code should use Generator (modify?)
        return X_bagged, Y_bagged

Let us create a forest with 400 trees, with a maximum depth of 12. We can train each tree on the same number of points as the training set (using bagging to add some randomness). Since we do not have that many features, we keep all 3 features

In [73]:
forest = Forest(10, 3, X.shape[0], 12)
forest.train(X_training, Y_training)

test_loss = 0
for i in range(len(Y_test)):
    predicted = forest.classify(X_test[i, :])
    if (Y_test[i] * predicted <= 0):
        test_loss += 1 / len(Y_test)
print('Test loss: ' + str(test_loss))

Test loss: 0.3191489361702114


Now we can try to see the effect of using Extra Trees. 

In [71]:
class ExtraTrees:

    def __init__(self, trees, n_features, n_samples, max_depth):
        self.forest = []
        self.features = n_features
        self.max_depth = max_depth
        self.samples = n_samples
        for i in range(trees):
            self.forest.append(Tree())

    def classify(self, x):
        vote = 0
        for tree in self.forest:
            vote += tree.classify(x)
        return np.sign(vote)

    def train(self, X, Y):
        for tree in self.forest:
            X_train, Y_train = self.bag(X, Y)
            tree.extra_training(X_train, Y_train, self.max_depth, False)
        
    def bag(self, X, Y):
        features = X.shape[1]
        points = X.shape[0]
        # Bagging: sample with replacement
        bagged = rnd.choices(range(points), k = self.samples)
        X_bagged = X[bagged, :]
        # Remove features that are not part of the tree
        selected = rnd.sample(range(features), k = self.features)
        for i in range(features):
            if i not in selected:
                X_bagged[:, i] = 0
        return X_bagged, Y[bagged]

Let us create an Extra Trees Classifer with 1000 trees, with a maximum depth of 12. We can train each tree on the same number of points as the training set (using bagging to add some randomness). Since we do not have that many features, we keep all 3 features

In [74]:
extra = ExtraTrees(10, 3, X.shape[0], 12)
extra.train(X_training, Y_training)

test_loss = 0
for i in range(len(Y_test)):
    predicted = extra.classify(X_test[i, :])
    if (Y_test[i] * predicted <= 0):
        test_loss += 1 / len(Y_test)
print('Test loss: ' + str(test_loss))

Test loss: 0.3340425531914876
